In [2]:
# ==============================================================
# Proyecto Final - Ciencia Computacional Intermedia (Tercer corte)
# Confiabilidad y planeación de capacidad de un servicio de inferencia de IA
# Grupo 1: Henry Cantor, Santiago Fajardo, Sebastián Páez, Sebastián Acuña
# ==============================================================
# Identificación del grupo (requisito del entregable E1).
# El grupo tiene 4 integrantes, así que cada C es la suma de las cédulas de una pareja:
#   C1 = Henry Cantor + Santiago Fajardo
#   C2 = Sebastián Páez + Sebastián Acuña
C1 = 2125981045
C2 = 2154025811
S = C1 + C2

print(f"C1 = {C1}")
print(f"C2 = {C2}")
print(f"S  = {S}")


C1 = 2125981045
C2 = 2154025811
S  = 4280006856


In [3]:
# Librerías permitidas por el enunciado (Sección 6.1)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats, optimize

# Única fuente de aleatoriedad de todo el notebook (requisito E1)
rng = np.random.default_rng(S)

# Parámetros personalizados del grupo (Sección 3 del enunciado)
tau = 180 + 10 * (S % 9)                    # umbral de latencia del SLO (ms)
epsilon = 0.01 * (1 + S % 3)                   # proporción máxima tolerada con L > tau
alpha = {0: 0.10, 1: 0.05, 2: 0.01}[S % 3]     # nivel de confianza = 1 - alpha
nivel_confianza = 1 - alpha
g = round(1.4 + 0.1 * (S % 7), 2)              # factor de crecimiento de la tasa de llegadas
t1_kb = 80 + 10 * (S % 5)                      # frontera pequeña / mediana (KB)
t2_kb = 400 + 25 * (S % 5)                     # frontera mediana / grande (KB)
pi_incidentes = (4 + S % 6) / 1000             # prevalencia de incidentes en producción (por minuto)
k_max = 4 + (S % 5)                            # máximo de falsas alarmas por día toleradas
r = S % 6                                      # desplazamiento entero (ítem B4)
N = 10 + (S % 11)                              # solicitudes consecutivas (ítem C4)
c0 = 2                                         # GPUs del pool real

parametros = pd.DataFrame({
    "Símbolo": ["tau (ms)", "epsilon", "1 - alpha", "g", "t1 (KB)", "t2 (KB)", "pi", "k_max", "r", "N", "c0"],
    "Valor": [tau, epsilon, nivel_confianza, g, t1_kb, t2_kb, pi_incidentes, k_max, r, N, c0],
})
parametros


,Símbolo,Valor
0,tau (ms),210.000
1,epsilon,0.010
2,1 - alpha,0.900
3,g,2.000
4,t1 (KB),90.000
5,t2 (KB),425.000
6,pi,0.004
7,k_max,5.000
8,r,0.000
9,N,14.000


In [4]:
# Estilo y utilidades comunes para cumplir las normas de la Sección 6.1:
# título, ejes con unidades, leyenda si hay más de una serie, resolución legible
# y un pie de figura que diga qué debe concluir el lector.
import os

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
})

CARPETA_FIGURAS = "figuras"
os.makedirs(CARPETA_FIGURAS, exist_ok=True)


def pie_de_figura(fig, numero, conclusion):
    """Agrega debajo de la figura el pie 'Figura n. <conclusión para el lector>'."""
    fig.text(0.5, -0.02, f"Figura {numero}. {conclusion}",
             ha="center", va="top", wrap=True, fontsize=10, style="italic")


def guardar_figura(fig, numero):
    """Guarda la figura como figuras/figura_<n>.png para insertarla en el informe."""
    ruta = os.path.join(CARPETA_FIGURAS, f"figura_{numero}.png")
    fig.savefig(ruta, bbox_inches="tight")
    return ruta


def formatear_intervalo(estimacion, inferior, superior, metodo, decimales=3):
    """Texto de un intervalo con estimación puntual, extremos, nivel y método (Sección 6.1)."""
    d = decimales
    return (f"{estimacion:.{d}f}  [{inferior:.{d}f}, {superior:.{d}f}]  "
            f"(IC {nivel_confianza:.0%}, método: {metodo})")


In [5]:
# Carga del paquete de datos único del grupo (entregado por el docente).
# Suban los CSV al panel de Archivos de Colab, o monten Google Drive y cambien RUTA_DATOS.
from pathlib import Path

RUTA_DATOS = Path(".")
ARCHIVOS = {
    "solicitudes": "solicitudes.csv",
    "monitoreo": "monitoreo.csv",
    # "validacion": "solicitudes_validacion.csv",  # se libera el día 9; solo se usa en el ítem F6
}

datos = {}
for nombre, archivo in ARCHIVOS.items():
    ruta = RUTA_DATOS / archivo
    if ruta.exists():
        datos[nombre] = pd.read_csv(ruta)
        print(f"{archivo}: {datos[nombre].shape[0]} filas x {datos[nombre].shape[1]} columnas")
    else:
        print(f"Falta {archivo} en {ruta.resolve().parent}")


Falta solicitudes.csv en /content
Falta monitoreo.csv en /content


# Parte A - Auditoría y preparación de los datos (8 pts)

### A1. Auditoría de `solicitudes.csv`
Inconsistencias encontradas y su tratamiento. Tabla "filas afectadas / criterio / tratamiento".

LECTURA DEL ARCHIVO

para ver si llgo como se esperaba

In [6]:
ARCHIVO = "solicitudes.csv"#archivo que se va a auditar.
T_VENTANA = 1200.0#duracion de la ventana de observacion
TOL_MS = None#tolerancia en ms para comparar tiempos
VERSIONES_VALIDAS = {"A", "B"}#etiquetas permitidas de la columna
COLS_NUM =["id_solicitud", "t_llegada_s", "tam_kb", "cache_hit", "red_ms",
           "gpu_ms", "latencia_ms", "intentos", "exito"]
COLS_ESPERADAS = ["id_solicitud", "t_llegada_s", "tam_kb", "cache_hit", "version", "red_ms",
                  "gpu_ms", "latencia_ms", "intentos", "exito"]
UMBRAL_REPETICION = 8#veces que se puede repetir un valor decimal antes de pensar que es centinela
FACTOR_REPETICION = 8 #debe repetirse este multiplo de lo que se repiten sus valores vecinos
TAM_FACTOR_MEDIANA = 25 #un tam_kb mayor a este multiplo es implausible
BYTES_POR_KB = 1024 #factor de conversion si el factor esta en bytes
ALPHA_EVIDENCIA = 0.05 #nivel de la prueba KS que respalda la coreccion de unidades

In [7]:
df_raw = pd.read_csv(ARCHIVO)#lee el archivo
df = df_raw.copy()#copia sobre la que operamos

faltan = [c for c in COLS_ESPERADAS if c not in df.columns]#columnas que faltan
sobran = [c for c in df.columns if c not in COLS_ESPERADAS]#columnas que sobran
print("formas(filas, columnas):", df.shape)#tamano del archivo
print("columnas que faltan:", faltan, "/ columnas extra:", sobran)
print("\ntipos de dato leidos por pandas")
print(df.dtypes)#columna marcada como object, significa texto mezclado o comas decimales

FileNotFoundError: [Errno 2] No such file or directory: 'solicitudes.csv'

In [ ]:
#conversion a numerico, sino se puede pasa a NaN y se cuenta
no_numericos = {}#guardamos esos valores
for c in COLS_NUM:#recorre cada columna que debe ser numerica
  antes = df[c].isna().sum()#ausentes que ya tenia la columna
  df[c] = pd.to_numeric(df[c], errors="coerce")#convierte a numero, el que no se pueda pasa NaN
  no_numericos[c] = int(df[c].isna().sum() - antes)#nuevos valores NaN
print("valores no numericos convertidos a NaN por columna:", no_numericos)

In [ ]:
print("ausentes por columna:")
print(df.isna().sum())#conteo de celdas vacias en cada columna
print("\nresumen numerico (minimos, cuantiles y maixmos):")
print(df[COLS_NUM].describe(percentiles=[.01, .5, .99]).T.round(2))
for c in ["version", "cache_hit", "intentos", "exito"]:
  print(f"\nfrecuencias de {c}:")
  print(df[c].value_counts(dropna=False))

como el archivo almacena tiempos redondeados (0,01ms), la regla L = red + espera + GPU se cumple unicamente bajo el redondeo. Medimos ese redondeo para declarar los errores.

In [ ]:
def decimales(s, maximo=6):
    """Mínimo número de decimales k tal que redondear la serie a k decimales no la cambia (hasta `maximo`)."""
    x = s.dropna().to_numpy(dtype=float)#valores presentes como arreglo numpy
    for k in range(maximo + 1):#prueba con 0,1,2.. decimales
      if np.allclose(np.round(x, k), x, rtol=0, atol=1e-9):#si redondeamos k decimales, queda igual?
        return k#el primer k que funciona es la resolucion del archivo
    return maximo#si ninguno funciona, el archivo esta bien de precision
DEC_TIEMPOS = max(decimales(df.red_ms), decimales(df.gpu_ms), decimales(df.latencia_ms))#decimales de los tiempos
RESOLUCION_MS = 10.0 ** -DEC_TIEMPOS  #resolucion dos decimales
if TOL_MS is None:#sino se fija a mano se deduce la resolucion
    TOL_MS = 1.5 * RESOLUCION_MS + 1e-9 #1,5 resoluciones + margen numerico diminuto
print(f"Decimales de los tiempos: {DEC_TIEMPOS} | resolución: {RESOLUCION_MS} ms | TOL_MS = {TOL_MS}")

In [ ]:
esp_tmp = df.latencia_ms - df.red_ms - df.gpu_ms #espera implicita, deberia de ser >=0
peq = esp_tmp[(esp_tmp < -1e-9) & (esp_tmp > -10 * RESOLUCION_MS)]#espera negativa (candidata a redondeo)
print("Esperas negativas pequeñas (|valor| < 10 resoluciones) y cuántas veces aparece cada una:")
print(peq.round(DEC_TIEMPOS + 2).value_counts().sort_index())#si salen uns poco valores fijos, va aredondeo
# Verificación complementaria: en filas con caché la espera es 0 por definición (L = red). Si el redondeo ya
# está «incluido» en ambos números, esa diferencia es 0 y NO delata el ruido
solo_cache = df[(df.cache_hit == 1) & (df.gpu_ms == 0)]#filas con cache y sin GPU
ruido = (solo_cache.latencia_ms - solo_cache.red_ms).abs()#tamano de la diferencia de esas filas
print("\nCuantiles de |latencia - red| en filas con caché (ms):")
print(ruido.quantile([.5, .9, .99, .999, 1]))#cuantiles altos pueden estar exgaerados por filas defectuosas

REGLAS DE VALIDACION

In [ ]:
def reglas(d):
    """Devuelve {codigo: (tipo, criterio, mascara)} con todas las reglas aplicadas al DataFrame d."""
    esp = d.latencia_ms - d.red_ms - d.gpu_ms                # espera en cola implícita (ms): L = red + espera + GPU, con espera >= 0
    r = {}                                                   # diccionario de resultados
    # --- duplicados -----------------------------------------------------------------------
    r["R01"] = ("error", "Fila duplicada exacta (copia de otra fila)",
                d.duplicated(keep="first"))                  # True en la 2.ª copia en adelante; la 1.ª se conserva
    r["R02"] = ("error", "id_solicitud repetido con contenido distinto",
                d.id_solicitud.duplicated(keep=False) & ~d.duplicated(keep=False))  # mismo id pero filas no idénticas
    # --- ausentes y no finitos ----------------------------------------------------------------
    for c in COLS_ESPERADAS:                                 # una regla por columna
        if c in d.columns:                                   # solo si la columna existe
            if c == "version":                               # texto: solo puede estar ausente
                m = d[c].isna()
            else:                                            # numéricas: ausente o infinito
                m = d[c].isna() | np.isinf(d[c])
            r[f"R03_{c}"] = ("error", f"{c} ausente, no numérico o infinito", m)
    # --- rangos físicos -----------------------------------------------------------------------
    r["R04"] = ("error", f"t_llegada_s fuera de [0, {T_VENTANA:.0f}] s",
                (d.t_llegada_s < 0) | (d.t_llegada_s > T_VENTANA))     # llegada fuera de la ventana observada
    r["R05"] = ("error", "tam_kb <= 0 (una imagen tiene tamaño positivo)", d.tam_kb <= 0)
    r["R18"] = ("error", f"tam_kb mayor que {TAM_FACTOR_MEDIANA} veces la mediana (posible error de unidades)",
                d.tam_kb > TAM_FACTOR_MEDIANA * d.tam_kb.median())     # un tamaño absurdamente grande respecto al resto
    r["R06"] = ("error", "red_ms <= 0 (un tiempo de red es positivo)", d.red_ms <= 0)
    r["R07"] = ("error", "gpu_ms < 0 (un tiempo no es negativo)", d.gpu_ms < 0)
    r["R16"] = ("error", "latencia_ms <= 0 (incluye el valor centinela -1)", d.latencia_ms <= 0)
    # --- dominios de variables discretas ------------------------------------------------------
    r["R08"] = ("error", "cache_hit distinto de 0 y 1", d.cache_hit.notna() & ~d.cache_hit.isin([0, 1]))
    r["R11"] = ("error", "version fuera de {A, B} (etiqueta mal escrita)", d.version.notna() & ~d.version.isin(VERSIONES_VALIDAS))
    r["R12"] = ("error", "intentos fuera de {1, 2, 3} (el SDK abandona tras 3)", d.intentos.notna() & ~d.intentos.isin([1, 2, 3]))
    r["R13"] = ("error", "exito distinto de 0 y 1", d.exito.notna() & ~d.exito.isin([0, 1]))
    # --- coherencia entre columnas -----------------------------------------------------------
    r["R09"] = ("error", "cache_hit = 1 pero gpu_ms > 0 (la caché no usa GPU)", (d.cache_hit == 1) & (d.gpu_ms > 0))
    r["R10"] = ("error", "cache_hit = 0 pero gpu_ms <= 0 (si usó GPU, G > 0)", (d.cache_hit == 0) & (d.gpu_ms <= 0))
    r["R14"] = ("error", "exito = 0 con menos de 3 intentos (abandona solo tras 3 fallos)", (d.exito == 0) & (d.intentos < 3))
    r["R15"] = ("error", "latencia < red + gpu más allá del redondeo (espera negativa)", esp < -TOL_MS)
    # --- observaciones informativas --------------------------------------------------------------
    r["R17"] = ("info", "Fila fuera de orden temporal (t menor que la fila anterior)", d.t_llegada_s.diff() < 0)
    return r

def tabla_auditoria(d):
    """Cuenta las filas que viola cada regla y devuelve la tabla de auditoría."""
    filas = []                                               # una fila de la tabla por cada regla
    for cod, (tipo, criterio, m) in reglas(d).items():
        filas.append({"regla": cod, "tipo": tipo, "criterio": criterio,
                      "filas_afectadas": int(m.sum()),       # cuántas filas violan la regla
                      "pct_del_total": round(100 * m.mean(), 3)})   # porcentaje sobre todas las filas
    return pd.DataFrame(filas)

aud = tabla_auditoria(df)                                    # auditoría del archivo tal como llegó
pd.set_option("display.width", 200, "display.max_colwidth", 90)   # que la tabla se vea completa en pantalla
print(aud[aud.filas_afectadas > 0].to_string(index=False))   # solo las reglas que sí encontraron algo
print("\nReglas sin hallazgos (revisadas y limpias):", list(aud[aud.filas_afectadas == 0].regla))

In [ ]:
# Filas afectadas por AL MENOS una regla de tipo error (una fila puede violar varias reglas a la vez)
mascaras = {k: v[2] for k, v in reglas(df).items() if v[0] == "error"}      # solo las reglas de tipo error
hay_error = pd.concat(mascaras, axis=1).any(axis=1)                          # True si la fila viola cualquiera
print(f"Filas con al menos un error: {hay_error.sum()} de {len(df)} ({100 * hay_error.mean():.2f} %)")
print("Suma de las columnas filas_afectadas:", int(aud[aud.tipo == 'error'].filas_afectadas.sum()),
      "(puede superar la cifra anterior porque hay filas que violan varias reglas)")

BUSQUEDA DE VALORES CENTINELA

In [ ]:
def repetidos_atipicos(s, dec, umbral, factor, vecinos=5):
    """Valores que aparecen >= umbral veces y >= factor veces la mediana de sus `vecinos` vecinos a cada lado."""
    res = 10.0 ** -dec                                       # distancia entre valores consecutivos posibles
    vc = s.dropna().round(dec).value_counts()                # cuántas veces aparece cada valor
    filas = []                                               # resultados
    for valor, n in vc[vc >= umbral].items():                # solo valores que se repiten lo suficiente
        vec = [vc.get(round(valor + k * res, dec), 0)        # veces que aparecen los valores vecinos
               for k in range(-vecinos, vecinos + 1) if k != 0]
        base = max(float(np.median(vec)), 1.0)               # repetición típica del entorno (mínimo 1)
        if n >= factor * base:                               # si el valor destaca mucho sobre su entorno
            filas.append((valor, int(n), base))
    return filas

sin_dup = df.drop_duplicates()                                # primero se quitan copias exactas, que repetirían todos los valores
reps = []                                                     # aquí se juntan los hallazgos
for c in ["tam_kb", "red_ms", "gpu_ms", "latencia_ms"]:       # columnas continuas
    dec = decimales(sin_dup[c])                               # resolución propia de cada columna
    for valor, n, base in repetidos_atipicos(sin_dup[c], dec, UMBRAL_REPETICION, FACTOR_REPETICION):
        reps.append({"columna": c, "valor": valor, "veces": n, "repeticion_tipica_vecinos": base})
reps = pd.DataFrame(reps)
print(reps.sort_values("veces", ascending=False).head(20).to_string(index=False) if len(reps) else "Sin valores atípicamente repetidos")
# Interpretación: gpu_ms = 0 es legítimo (caché). Cualquier otro valor que destaque (p. ej. -1 en latencia_ms) merece revisión.

EVIDENCIA QUE NOS AYUDA PARA DECIDIR EL TRATAMIENTO.

In [ ]:
# -a. R09: caché con GPU -> ¿qué campo es el erróneo? (se usa la identidad L = red + espera + GPU)
c9 = df[(df.cache_hit == 1) & (df.gpu_ms > 0)].copy()        # filas de la regla R09
c9["dif_lat_red"] = (c9.latencia_ms - c9.red_ms).abs()       # si es ~0, la latencia es la de una respuesta SIN GPU
c9["dif_lat_red_gpu"] = (c9.latencia_ms - c9.red_ms - c9.gpu_ms).abs()   # si es ~0, la latencia sí incluye la GPU
culpa_gpu = (c9.dif_lat_red <= TOL_MS)                       # la latencia coincide con «sin GPU»: el campo erróneo es gpu_ms
culpa_cache = (c9.dif_lat_red_gpu <= TOL_MS) & ~culpa_gpu    # la latencia incluye la GPU: el campo erróneo es cache_hit
print(f"R09: {len(c9)} filas | gpu_ms es el campo erróneo: {int(culpa_gpu.sum())} | "
      f"cache_hit es el campo erróneo: {int(culpa_cache.sum())} | sin evidencia: {int((~culpa_gpu & ~culpa_cache).sum())}")

In [ ]:
# -b. R18: tam_kb implausible -> ¿es un error de unidades (bytes en vez de KB)?
umbral_tam = TAM_FACTOR_MEDIANA * df.tam_kb.median()          # umbral fijo de implausibilidad (se usará también al limpiar)
grandes = df[df.tam_kb > umbral_tam].drop_duplicates()        # filas implausibles (sin copias repetidas)
normales = df.loc[df.tam_kb <= umbral_tam, "tam_kb"]          # tamaños normales, para comparar
DEC_TAM = decimales(normales)                                 # resolución con la que se guardan los tamaños normales
CORREGIR_TAM = False                                          # por defecto no se corrige; la evidencia puede cambiarlo
print(f"Umbral de implausibilidad: {umbral_tam:.1f} KB | filas implausibles: {len(grandes)} | "
      f"mayor tamaño normal: {normales.max():.1f} KB | menor tamaño implausible: "
      f"{grandes.tam_kb.min() if len(grandes) else float('nan'):.1f} KB")
if len(grandes) > 0:
    q = grandes.tam_kb / BYTES_POR_KB                         # tamaños implausibles reinterpretados como bytes -> KB
    frac_exacta = float((np.abs(np.round(q, DEC_TAM) - q) < 1e-6).mean())   # ¿al dividir quedan con la misma resolución que los normales?
    ks_p = float(stats.ks_2samp(q, normales.dropna()).pvalue)   # ¿se distribuyen como los tamaños normales? (p alto = sí)
    print(f"Tras dividir entre {BYTES_POR_KB}: fracción con exactamente {DEC_TAM} decimales = {frac_exacta:.3f} "
          f"| mediana {q.median():.1f} KB frente a {normales.median():.1f} KB normales | KS p = {ks_p:.3f}")
    print("Reparto de las filas implausibles por versión:", grandes.version.value_counts().to_dict(),
          "| por caché:", grandes.cache_hit.value_counts().to_dict())
    CORREGIR_TAM = (frac_exacta >= 0.95) and (ks_p > ALPHA_EVIDENCIA)    # criterio de decisión explícito
print("¿Se corrige dividiendo entre", BYTES_POR_KB, "? ->", CORREGIR_TAM)

In [ ]:
# -c. R03: ausentes que se pueden deducir con la lógica del sistema (solo en filas con caché)
rn = df[df.red_ms.isna()]                                     # filas sin tiempo de red
gn = df[df.gpu_ms.isna()]                                     # filas sin tiempo de GPU
rec_red = int(((rn.cache_hit == 1) & (rn.latencia_ms > 0)).sum())   # con caché, L = red, así que red = latencia
rec_gpu = int((gn.cache_hit == 1).sum())                      # con caché, gpu_ms = 0 por definición
print(f"red_ms ausente: {len(rn)} filas, deducibles (caché y latencia válida): {rec_red}, no deducibles: {len(rn) - rec_red}")
print(f"gpu_ms ausente: {len(gn)} filas, deducibles (caché): {rec_gpu}, no deducibles: {len(gn) - rec_gpu}")

TRATAMIENTO

In [ ]:
TRATAMIENTO = {                                              # código de regla -> qué se hace con las filas que la violan
    "R01": "eliminar la copia",                              # una solicitud ocurre una sola vez
    "R02": "conservar la copia consistente con la identidad de tiempos si hay exactamente una; si no, eliminar todas",
    "R03": "corregir si es deducible (caché); si no, eliminar",   # ausentes: solo se corrige lo que la lógica permite deducir
    "R04": "eliminar", "R05": "eliminar", "R06": "eliminar", "R07": "eliminar", "R16": "eliminar",  # valores físicamente imposibles
    "R08": "eliminar", "R12": "eliminar", "R13": "eliminar", # dominios inválidos sin forma de recuperar el valor correcto
    "R09": "corregir gpu_ms = 0 si la latencia coincide con «sin GPU»; si no, eliminar",  # decisión basada en la evidencia 5a
    "R10": "eliminar",                                       # no se puede recuperar el tiempo de GPU
    "R11": "normalizar la etiqueta (quitar espacios y pasar a mayúscula); si sigue inválida, eliminar",
    "R14": "eliminar",                                       # no se sabe si el error está en exito o en intentos
    "R15": "eliminar",                                       # la identidad no se cumple más allá del redondeo y no hay cómo repararla
    "R17": "conservar (informativa); se ordena por tiempo al final",
    "R18": "dividir entre 1024 si la evidencia 5b lo respalda; si no, eliminar",   # error de unidades (bytes en vez de KB)
}

In [ ]:
def limpiar(d):
    """Aplica el tratamiento en orden y devuelve (d_limpio, flujo), donde flujo registra cuántas filas se pierde en cada paso."""
    d = d.copy()                                             # no modificar el DataFrame de entrada
    flujo = [("Filas iniciales", len(d))]                    # lista de (paso, filas restantes)

    d = d[~d.duplicated(keep="first")]                       # paso 1: eliminar copias exactas ANTES de contar otros defectos
    flujo.append(("Quitar duplicados exactos (R01)", len(d)))

    d["version"] = d.version.astype("string").str.strip().str.upper()   # paso 2: normalizar la etiqueta (espacios y mayúsculas)
    flujo.append(("Normalizar etiqueta version (R11), sin perder filas", len(d)))

    # paso 3: correcciones que la lógica del sistema permite DEDUCIR (no se pierden filas)
    cond9 = (d.cache_hit == 1) & (d.gpu_ms > 0) & ((d.latencia_ms - d.red_ms).abs() <= TOL_MS)   # R09 con evidencia a favor de gpu_ms
    d.loc[cond9, "gpu_ms"] = 0.0                             # la caché no usa GPU
    flujo.append((f"Corregir gpu_ms = 0 en caché (R09), {int(cond9.sum())} filas, sin perder filas", len(d)))

    if CORREGIR_TAM:                                         # R18: solo si la evidencia de la sección 5b lo respaldó
        cond18 = d.tam_kb > umbral_tam                       # filas con tamaño implausible
        d.loc[cond18, "tam_kb"] = (d.loc[cond18, "tam_kb"] / BYTES_POR_KB).round(DEC_TAM)   # bytes -> KB
        flujo.append((f"Corregir tam_kb dividiendo entre {BYTES_POR_KB} (R18), {int(cond18.sum())} filas, sin perder filas", len(d)))

    c_gpu = d.gpu_ms.isna() & (d.cache_hit == 1)             # R03: caché y gpu_ms ausente
    d.loc[c_gpu, "gpu_ms"] = 0.0                             # con caché, gpu_ms = 0 por definición
    c_red = d.red_ms.isna() & (d.cache_hit == 1) & (d.latencia_ms > 0)   # R03: caché, red ausente y latencia válida
    d.loc[c_red, "red_ms"] = d.loc[c_red, "latencia_ms"]     # con caché, L = red
    flujo.append((f"Deducir ausentes en filas con caché (R03): gpu_ms {int(c_gpu.sum())}, red_ms {int(c_red.sum())}, sin perder filas", len(d)))

    # paso 4: ids repetidos con contenido distinto (R02): quedarse con la copia sana si hay exactamente una
    rg = reglas(d)                                           # reglas sobre el estado actual
    mala = pd.concat({k: v[2] for k, v in rg.items()
                      if v[0] == "error" and k not in ("R01", "R02")}, axis=1).any(axis=1)   # fila con algún otro error
    dup_id = d.id_solicitud.duplicated(keep=False)           # filas cuyo id aparece más de una vez
    sanas = (dup_id & ~mala).astype(int).groupby(d.id_solicitud).transform("sum")   # cuántas copias sanas tiene cada id
    quitar = dup_id & mala & (sanas == 1)                    # copia defectuosa cuando existe exactamente una sana
    d = d[~quitar]
    flujo.append((f"Resolver ids repetidos (R02): quitar la copia defectuosa, {int(quitar.sum())} filas", len(d)))

    # paso 5: eliminar las filas que aún violen alguna regla de tipo error.
    # Las máscaras se calculan UNA vez sobre el estado actual de d (así todas quedan alineadas) y luego se descartan juntas.
    ya = pd.Series(False, index=d.index)                     # marca las filas ya condenadas por una regla anterior
    costo = []                                               # (regla, filas nuevas que elimina esa regla)
    for cod, (tipo, criterio, m) in reglas(d).items():       # recorre las reglas en su orden de definición
        if tipo == "error":                                  # las informativas (R17) no eliminan nada
            nuevas = int((m & ~ya).sum())                    # filas que esta regla condena y que ninguna anterior había condenado
            if nuevas > 0:
                costo.append((f"Eliminar por {cod}: {criterio}", nuevas))
            ya = ya | m                                      # acumula las filas condenadas
    d = d[~ya]                                               # descarta todas de una vez
    restantes = len(d) + sum(n for _, n in costo)            # filas antes de este paso, para ir restando una regla a la vez
    for texto, nuevas in costo:                              # registra el costo de cada regla en el flujo
        restantes -= nuevas
        flujo.append((texto, restantes))

    d = d.sort_values("t_llegada_s", kind="stable").reset_index(drop=True)   # orden temporal, requerido por los análisis siguientes
    flujo.append(("Ordenar por t_llegada_s, sin perder filas", len(d)))
    return d, pd.DataFrame(flujo, columns=["paso", "filas_restantes"])

df_limpio, flujo = limpiar(df)                               # ejecuta la limpieza
flujo["filas_perdidas"] = (-flujo.filas_restantes.diff()).fillna(0).astype(int)   # filas que se pierde en cada paso
print(flujo.to_string(index=False))

VOLVEMOS A AUDITAR PARA VERIFICAR

In [ ]:
aud_post = tabla_auditoria(df_limpio)                        # auditoría del archivo ya limpio
residual = aud_post[(aud_post.tipo == "error") & (aud_post.filas_afectadas > 0)]   # reglas de error que aún se violan
print("Reglas de error que aún se violan:\n", residual.to_string(index=False) if len(residual) else "ninguna")
assert len(residual) == 0, "La limpieza no eliminó todas las violaciones: revisa limpiar()"   # frena el notebook si algo quedó mal
print(f"\nFilas: {len(df_raw)} (original) -> {len(df_limpio)} (limpio) | perdidas: {len(df_raw) - len(df_limpio)} "
      f"({100 * (len(df_raw) - len(df_limpio)) / len(df_raw):.2f} %)")

IMPACTO DE LA LIMPIEZA

In [ ]:
def resumen(d, nombre):
    """Estadísticos clave de un DataFrame de solicitudes."""
    nc = d[(d.cache_hit == 0)]                               # solicitudes que usaron GPU
    return pd.Series({"filas": len(d),
                      "latencia media (ms)": d.latencia_ms.mean(),
                      "latencia mediana (ms)": d.latencia_ms.median(),
                      "latencia p95 (ms)": d.latencia_ms.quantile(.95),
                      "P(L > tau)": (d.latencia_ms > tau).mean(),     # fracción de solicitudes por encima del umbral del SLO
                      "tasa de caché": d.cache_hit.mean(),
                      "GPU media sin caché (ms)": nc.gpu_ms.mean(),
                      "tam_kb medio": d.tam_kb.mean(),
                      "red media (ms)": d.red_ms.mean()}, name=nombre)

impacto = pd.concat([resumen(df, "original"), resumen(df_limpio, "limpio")], axis=1)   # lado a lado
impacto["cambio %"] = 100 * (impacto["limpio"] - impacto["original"]) / impacto["original"]  # variación relativa
print(impacto.round(4).to_string())
# Nota: el tam_kb medio del original está inflado por los valores en bytes; el cambio grande en esa fila es esperable y es la prueba del defecto.

GRAFICA PARA EL INFORME

In [ ]:
aud["tratamiento"] = aud.regla.str[:3].map(TRATAMIENTO)       # une cada regla con su tratamiento (R03_x se agrupa como R03)
aud.to_csv("A1_tabla_auditoria.csv", index=False)             # tabla «filas afectadas / criterio / tratamiento» (anexo del informe)
flujo.to_csv("A1_flujo_filas.csv", index=False)               # cuántas filas se pierde en cada paso (bitácora)
df_limpio.to_csv("solicitudes_limpio.csv", index=False)       # archivo limpio que usarán A2 y las demás partes

hall = aud[aud.filas_afectadas > 0].sort_values("filas_afectadas")      # solo reglas con hallazgos, de menor a mayor
fig, ax = plt.subplots(figsize=(8, 0.45 * len(hall) + 1.5))   # alto proporcional al número de barras
ax.barh(hall.regla, hall.filas_afectadas)                     # barras horizontales: una por regla
ax.set_xlabel("Filas afectadas (de todo el archivo)")         # unidades del eje
ax.set_title("A1: inconsistencias encontradas por regla")     # título
for i, v in enumerate(hall.filas_afectadas):                  # escribe el número al final de cada barra
    ax.text(v, i, f" {v}", va="center")
plt.tight_layout()
plt.savefig("fig_A1_defectos.png", dpi=150)                   # guarda la figura para el anexo
plt.show()

In [ ]:
# Resumen en texto para la bitácora
print(f"A1 · archivo {ARCHIVO}: {len(df_raw)} filas, {df_raw.shape[1]} columnas.")
print(f"A1 · filas con al menos un error: {int(hay_error.sum())} ({100 * hay_error.mean():.2f} %).")
print(f"A1 · filas finales tras limpieza: {len(df_limpio)} (perdidas: {len(df_raw) - len(df_limpio)}).")
print(f"A1 · resolución de tiempos = {RESOLUCION_MS} ms | TOL_MS = {TOL_MS} | corrección de unidades en tam_kb = {CORREGIR_TAM}")
print(f"A1 · UMBRAL_REPETICION = {UMBRAL_REPETICION} | FACTOR_REPETICION = {FACTOR_REPETICION} | T_VENTANA = {T_VENTANA}")

### A2. ¿Un único régimen?
Criterio estadístico cuantitativo, segmentos, sensibilidad al criterio y régimen estable que se usará en adelante.

**Paso 1. ¿Toda la ventana es un único régimen?**

Si la tasa de llegadas fuera constante (Poisson homogéneo), los conteos en bins de 10 s tendrían varianza ≈ media. El índice de dispersión $D=\sum (c_i-\bar c)^2/\bar c$ sigue aprox. una $\chi^2_{k-1}$.

In [ ]:
d_a2 = df_limpio.sort_values("t_llegada_s").reset_index(drop=True)   # orden temporal
conteos = np.histogram(d_a2.t_llegada_s, bins=np.arange(0, T_VENTANA + 10, 10))[0]   # llegadas cada 10 s

D = ((conteos - conteos.mean())**2 / conteos.mean()).sum()     # índice de dispersión
p_disp = stats.chi2.sf(D, len(conteos) - 1)                     # valor p bajo Poisson homogéneo
print(f"D = {D:.1f}, gl = {len(conteos) - 1}, p = {p_disp:.2e}, varianza/media = {conteos.var(ddof=1) / conteos.mean():.1f}")
print("-> NO es un único régimen" if p_disp < alpha else "-> Compatible con un único régimen")

**Paso 2. ¿Dónde empieza y termina el régimen distinto? (y sensibilidad al criterio)**

Se prueban todos los cortes posibles (inicio, fin) y se elige el que maximiza la verosimilitud de un modelo con tres tramos de tasa constante. Se repite con varios anchos de bin y con otra variable (reintentos) para ver si los cortes cambian.

In [ ]:
def buscar_rafaga(c, ancho):
    """Devuelve (inicio_s, fin_s) del tramo que mejor se separa del resto (máx. verosimilitud Poisson)."""
    def logver(tramo):                                     # log-verosimilitud de un tramo con tasa constante
        s, L = tramo.sum(), len(tramo)
        return s * np.log(s / L) - s if s > 0 else 0
    mejor, corte = -np.inf, None
    for i in range(1, len(c)):                             # todos los pares de cortes i < j
        for j in range(i + 1, len(c)):
            v = logver(c[:i]) + logver(c[i:j]) + logver(c[j:])
            if v > mejor:
                mejor, corte = v, (i * ancho, j * ancho)
    return corte

# Criterio 1: llegadas, con distintos anchos de bin
for ancho in [5, 10, 20, 30, 60]:
    llegadas = np.histogram(d_a2.t_llegada_s, bins=np.arange(0, T_VENTANA + ancho, ancho))[0]
    print(f"Llegadas, bin {ancho:2d} s -> ráfaga en {buscar_rafaga(llegadas, ancho)}")

# Criterio 2: solicitudes que necesitaron reintento (otra variable, misma idea)
reintentos = np.histogram(d_a2.t_llegada_s[d_a2.intentos > 1], bins=np.arange(0, T_VENTANA + 10, 10))[0]
print(f"Reintentos, bin 10 s -> ráfaga en {buscar_rafaga(reintentos, 10)}")

**Paso 3. Régimen estable y verificación**

Se quita la ráfaga. Para unir el tramo anterior y el posterior se comprueba que no difieren (4 pruebas a la vez, por eso se usa Bonferroni: α/4).

In [ ]:
INI, FIN = 370, 550                                            # cortes hallados en el paso 2
en_rafaga = d_a2.t_llegada_s.between(INI, FIN, inclusive="left")
df_estable, df_rafaga = d_a2[~en_rafaga].copy(), d_a2[en_rafaga].copy()   # df_estable se usa en B a F
pre  = df_estable[df_estable.t_llegada_s < INI]
post = df_estable[df_estable.t_llegada_s >= FIN]

# Resumen por tramo
for nombre, d, dur in [("antes", pre, INI), ("ráfaga", df_rafaga, FIN - INI), ("después", post, T_VENTANA - FIN)]:
    print(f"{nombre:8s} tasa = {len(d) / dur:5.2f} sol/s | P(intentos>1) = {(d.intentos > 1).mean():.3f} | "
          f"latencia media = {d.latencia_ms.mean():.1f} ms | éxito = {d.exito.mean():.3f}")

# ¿Antes y después son iguales?
pruebas = {
    "tasa":       stats.binomtest(len(pre), len(df_estable), INI / (T_VENTANA - (FIN - INI))).pvalue,
    "reintentos": stats.chi2_contingency(pd.crosstab(df_estable.t_llegada_s >= FIN, df_estable.intentos > 1))[1],
    "latencia":   stats.ks_2samp(pre.latencia_ms, post.latencia_ms).pvalue,
    "gpu":        stats.ks_2samp(pre.gpu_ms[pre.cache_hit == 0], post.gpu_ms[post.cache_hit == 0]).pvalue,
}
print(f"\nAntes vs después (Bonferroni, α/4 = {alpha / 4:.3f}):")
for nombre, pv in pruebas.items():
    print(f"  {nombre:11s} p = {pv:.3f} -> {'difieren' if pv < alpha / 4 else 'iguales'}")

# ¿El régimen estable es Poisson homogéneo?
c_est = np.concatenate([np.histogram(pre.t_llegada_s, np.arange(0, INI + 1, 10))[0],
                        np.histogram(post.t_llegada_s, np.arange(FIN, T_VENTANA + 1, 10))[0]])
D_est = ((c_est - c_est.mean())**2 / c_est.mean()).sum()
print(f"\nRégimen estable: D = {D_est:.1f}, p = {stats.chi2.sf(D_est, len(c_est) - 1):.3f} | "
      f"{len(df_estable)} solicitudes (ráfaga: {len(df_rafaga)})")

**Conclusión A2.** La ventana no es un único régimen: hay una ráfaga en [370, 550) s donde la tasa de llegadas y los reintentos se duplican. Los cortes no dependen del ancho del bin ni de la variable usada. El régimen estable es [0, 370) ∪ [550, 1200) s (`df_estable`), y es el que se usa en las partes B a F.

### A3. Resumen descriptivo
Centro, dispersión, asimetría y curtosis de latencia_ms, gpu_ms, red_ms y tam_kb, por versión y por régimen.

**Figura 1:** evolución temporal del sistema.

**Cómo se resume.** Se usan los datos limpios de A1 con la etiqueta de régimen de A2 (`estable` = [0, 370) ∪ [550, 1200) s, `rafaga` = [370, 550) s).
Para cada variable se reporta, por régimen y por versión (A, B y A+B):
- **centro:** media y mediana
- **dispersión:** desviación estándar e IQR
- **forma:** asimetría y curtosis en exceso (0 = como la normal)

`gpu_ms` se reporta dos veces: con todas las solicitudes y solo con las no cacheadas, porque las cacheadas tienen `gpu_ms = 0` y ese bloque de ceros distorsiona la forma (la variable mixta se estudia en C1).

In [ ]:
# Paso 1. Tabla descriptiva por variable, régimen y versión
d_a2["regimen"] = np.where(d_a2.t_llegada_s.between(INI, FIN, inclusive="left"), "rafaga", "estable")   # etiqueta de A2
d_a2["gpu_sin_cache"] = d_a2.gpu_ms.where(d_a2.cache_hit == 0)   # GPU solo de las solicitudes que usaron GPU

VARIABLES_A3 = ["latencia_ms", "gpu_ms", "gpu_sin_cache", "red_ms", "tam_kb"]


def descriptivos(x):
    """Centro, dispersión, asimetría y curtosis en exceso de una muestra."""
    x = x.dropna()
    return pd.Series({"n": len(x),
                      "media": x.mean(), "mediana": x.median(),                        # centro
                      "desv_est": x.std(), "IQR": x.quantile(.75) - x.quantile(.25),   # dispersión
                      "asimetria": stats.skew(x),                                      # > 0: cola a la derecha
                      "curtosis_exceso": stats.kurtosis(x)})                           # 0: como la normal


con_total = pd.concat([d_a2, d_a2.assign(version="A+B")])       # duplica las filas como "A+B" para tener el total
tabla_a3 = (con_total.melt(id_vars=["regimen", "version"], value_vars=VARIABLES_A3, var_name="variable")
            .groupby(["variable", "regimen", "version"])["value"].apply(descriptivos).unstack()
            .reindex(VARIABLES_A3, level="variable"))            # mismo orden de VARIABLES_A3
tabla_a3["n"] = tabla_a3["n"].astype(int)
tabla_a3.round(4).to_csv("A3_resumen_descriptivo.csv")         # anexo del informe
tabla_a3.round(2)

**Paso 2. ¿Qué cambia?** Se comparan las medias en dos direcciones: ráfaga frente a estable, y versión B frente a A. También se calcula la espera en cola implícita (L − red − GPU), que explica por qué sube la latencia. Es solo descriptivo: la inferencia con intervalos viene en F3.

In [ ]:
# Paso 2. Cambio relativo de la media (%) y contexto por régimen
m = tabla_a3["media"]                                           # medias de la tabla del paso 1
cambios = pd.DataFrame({
    "ráfaga vs estable (%)": 100 * (m.xs(("rafaga", "A+B"), level=[1, 2]) / m.xs(("estable", "A+B"), level=[1, 2]) - 1),
    "B vs A, estable (%)":   100 * (m.xs(("estable", "B"), level=[1, 2]) / m.xs(("estable", "A"), level=[1, 2]) - 1),
    "B vs A, ráfaga (%)":    100 * (m.xs(("rafaga", "B"), level=[1, 2]) / m.xs(("rafaga", "A"), level=[1, 2]) - 1),
})
print(cambios.round(1).to_string(), "\n")

d_a2["espera_ms"] = (d_a2.latencia_ms - d_a2.red_ms - d_a2.gpu_ms).clip(lower=0)   # L = red + espera + GPU
duracion = {"estable": T_VENTANA - (FIN - INI), "rafaga": FIN - INI}               # segundos de cada régimen
contexto = d_a2.groupby("regimen").agg(n=("t_llegada_s", "size"),
                                       espera_media_ms=("espera_ms", "mean"),
                                       tasa_cache=("cache_hit", "mean"),
                                       P_L_mayor_tau=("latencia_ms", lambda s: (s > tau).mean()),
                                       P_intentos_mayor_1=("intentos", lambda s: (s > 1).mean()))
contexto.insert(1, "llegadas_por_s", contexto.n / pd.Series(duracion))
print(f"Contexto por régimen (tau = {tau} ms):")
print(contexto.round(4).to_string())

### Figura 1. Evolución temporal del sistema
Cuatro paneles con el mismo eje de tiempo (bins de 10 s) y la ráfaga sombreada: carga (llegadas), latencia por versión frente a τ, reintentos y tiempo de GPU. Así se ve qué cambia con el régimen y qué no.

In [ ]:
# Paso 3. Figura 1
ANCHO = 10                                                      # segundos por bin
bin_s = (d_a2.t_llegada_s // ANCHO) * ANCHO + ANCHO / 2        # centro del bin de cada solicitud
por_bin = d_a2.groupby(bin_s)
por_bin_ver = d_a2.groupby([bin_s, "version"])
COLOR = {"A": "tab:blue", "B": "tab:orange"}

fig, ax = plt.subplots(4, 1, figsize=(10, 12), sharex=True)

ax[0].plot(por_bin.size() / ANCHO, color="black", label="Llegadas")
ax[0].set_ylabel("Llegadas (solicitudes/s)")
ax[0].set_title("Figura 1. Evolución temporal del sistema (bins de 10 s)")

ax[1].plot(por_bin.latencia_ms.quantile(0.95), color="grey", linestyle="--", label="p95 (A+B)")
lat = por_bin_ver.latencia_ms.mean().unstack()
rein = por_bin_ver.intentos.apply(lambda s: (s > 1).mean()).unstack()
gpu = por_bin_ver.gpu_sin_cache.mean().unstack()
for v in ["A", "B"]:
    ax[1].plot(lat[v], color=COLOR[v], label=f"Media, versión {v}")
    ax[2].plot(rein[v], color=COLOR[v], label=f"Versión {v}")
    ax[3].plot(gpu[v], color=COLOR[v], label=f"Versión {v}")
ax[1].axhline(tau, color="red", linestyle=":", label=f"Umbral τ = {tau} ms")
ax[1].set_ylabel("Latencia (ms)")
ax[2].set_ylabel("P(intentos > 1)")
ax[3].set_ylabel("GPU media sin caché (ms)")
ax[3].set_xlabel("Tiempo desde el inicio de la ventana (s)")

for a_ in ax:                                                   # ráfaga sombreada y leyenda en cada panel
    a_.axvspan(INI, FIN, color="grey", alpha=0.2, label=f"Ráfaga [{INI}, {FIN}) s")
    a_.margins(y=0.22)
    a_.legend(loc="upper right", fontsize=8, ncol=3)
    a_.set_xlim(0, T_VENTANA)

frac_B_menor = (gpu["B"] < gpu["A"]).mean()                    # bins donde B tiene menos GPU que A
est, raf = contexto.loc["estable"], contexto.loc["rafaga"]
pie_de_figura(fig, 1,
              f"En la ráfaga las llegadas pasan de {est.llegadas_por_s:.0f} a {raf.llegadas_por_s:.0f} sol/s, la latencia media "
              f"sube {cambios.loc['latencia_ms', 'ráfaga vs estable (%)']:.0f} % y P(L > τ) va de {100 * est.P_L_mayor_tau:.1f} % "
              f"a {100 * raf.P_L_mayor_tau:.1f} %;\nel tiempo de GPU no cambia con el régimen, y la versión B usa "
              f"{abs(cambios.loc['gpu_sin_cache', 'B vs A, estable (%)']):.0f} % menos GPU que la A ({100 * frac_B_menor:.0f} % de los bins).")
plt.tight_layout()
guardar_figura(fig, 1)                                          # figuras/figura_1.png
plt.show()

**Conclusión A3.**
- **La ráfaga solo afecta la latencia:** la media sube ≈ 54 % y la espera en cola pasa de ≈ 3 a ≈ 32 ms, mientras GPU, red y tamaño no cambian. El deterioro es por carga, no por el modelo.
- **La versión B usa ≈ 6 % menos GPU que la A** en ambos regímenes. Es una pista descriptiva; la prueba formal es F3.
- **Forma:** latencia, GPU y tamaño son asimétricas a la derecha (media > mediana, asimetría > 0) y con colas pesadas; `red_ms` tiene asimetría y curtosis ≈ 0, compatible con una normal (se prueba en C3).

# Parte B - Llegadas, fallos y reintentos (14 pts)

### B1. Modelo del proceso de llegadas

### B2. Bloques de 100 solicitudes

### B3. Modelo de la variable intentos

### B4. Reintentos en una ventana de 10 s

# Parte C - Tiempo de cómputo y tiempo de red (14 pts)

### C1. Distribución de G (todas las solicitudes)

### C2. Máxima verosimilitud para la parte continua de G

### C3. ¿red_ms admite un modelo normal?

### C4. Tiempo total de GPU de N solicitudes no cacheadas

# Parte D - Dependencias y esperanza condicional (14 pts)

### D1. Conjunta (clase de tamaño, cache_hit)

### D2. Esperanza condicional

### D3. Modelo lineal para E[G | tam_kb, no caché]

### D4. Covarianza y correlación entre tam_kb y gpu_ms (no cacheadas)

# Parte E - Monitoreo y decisión de alertas (12 pts)

### E1. Sensibilidad y tasa de falsos positivos

### E2. Probabilidad de incidente dada cada regla en producción

### E3. ¿D1 y D2 son condicionalmente independientes?

### E4. Falsas alarmas por día

# Parte F - Inferencia, distribuciones muestrales y comparación de versiones (16 pts)

### F1. Distribución muestral de la media de latencia_ms

### F2. Intervalo para la latencia media

### F3. Versión B frente a versión A

### F4. Cobertura de intervalos para una proporción

### F5. Tamaño de muestra por versión

### F6. Validación con solicitudes_validacion.csv (día 9)

# Parte G - Simulación y decisión de capacidad (14 pts)

### G1. Simulador de Montecarlo con c GPUs

### G2. Capacidad con la tasa de llegadas multiplicada por g

### G3. Robustez de c*

### G4. Memorando ejecutivo
Se entrega aparte como PDF de una página (E4).